# CNN — Classificação de Formato de Rosto

Experimento exploratório com EfficientNetB2 para classificar 5 formatos de rosto (Heart, Oblong, Oval, Round, Square) e recomendar óculos ideais.

**Dataset:** Kaggle Face Shape Dataset  
**Arquitetura:** EfficientNetB2 (transfer learning, base descongelada)  
**Melhor resultado:** 85% de acurácia no conjunto de teste  

---

## 1. Imports e configuração

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Verifica GPU
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

## 2. Base de dados

In [ ]:
TRAINING_DIR = "data/training_set"
TEST_DIR = "data/testing_set"

# Visualiza uma imagem de exemplo
img_exemplo = os.path.join(TRAINING_DIR, "Heart/heart (1).jpg")
tf.keras.preprocessing.image.load_img(img_exemplo)

In [ ]:
# Distribuição de imagens por classe
print("Imagens por classe (treino):")
for classe in sorted(os.listdir(TRAINING_DIR)):
    total = len(os.listdir(f"{TRAINING_DIR}/{classe}"))
    print(f"  {classe}: {total}")

### 2.1 Geradores de dados

- **Treino:** augmentation completo (rotação, zoom, flip, brilho)
- **Teste:** apenas normalização via `preprocess_input`

In [ ]:
gerador_treinamento = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

gerador_teste = ImageDataGenerator(preprocessing_function=preprocess_input)

dataset_treinamento = gerador_treinamento.flow_from_directory(
    TRAINING_DIR,
    target_size=(288, 288),
    batch_size=32,
    class_mode="categorical",
    shuffle=True,
    seed=42,
)

dataset_teste = gerador_teste.flow_from_directory(
    TEST_DIR,
    target_size=(288, 288),
    batch_size=32,
    class_mode="categorical",
    shuffle=False,
)

print(f"\nTreino:  {dataset_treinamento.samples} imagens")
print(f"Teste:   {dataset_teste.samples} imagens")
print(f"Classes: {dataset_treinamento.class_indices}")

## 3. Modelo — EfficientNetB2

Estratégia: base **totalmente descongelada** desde o início.
Essa abordagem obteve **85% de acurácia** no dataset Kaggle (4.000 imagens).

> **Nota:** experimentos anteriores com base congelada + fine-tuning resultaram em ~50-53%.  
> Descongelar a base inteira com `learning_rate=1e-4` foi o principal ganho.

In [ ]:
modelo_base = keras.applications.EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(288, 288, 3),
    pooling="avg",
)
modelo_base.trainable = True

print(f"Total de camadas: {len(modelo_base.layers)}")

### 3.1 Cabeça de classificação

In [ ]:
x = keras.layers.BatchNormalization()(modelo_base.output)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.5)(x)
output = keras.layers.Dense(
    len(dataset_treinamento.class_indices), activation="softmax"
)(x)

modelo_treinado = keras.Model(inputs=modelo_base.input, outputs=output)

modelo_treinado.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

modelo_treinado.summary()

## 4. Treinamento

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=7, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="best_model_kaggle.keras", monitor="val_loss", save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history = modelo_treinado.fit(
    dataset_treinamento,
    epochs=100,
    validation_data=dataset_teste,
    callbacks=callbacks,
)

### 4.1 Curvas de aprendizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["accuracy"], label="Treino")
axes[0].plot(history.history["val_accuracy"], label="Validação")
axes[0].set_title("Acurácia")
axes[0].set_xlabel("Época")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Treino")
axes[1].plot(history.history["val_loss"], label="Validação")
axes[1].set_title("Loss")
axes[1].set_xlabel("Época")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Avaliação

In [ ]:
previsoes = modelo_treinado.predict(dataset_teste)
y_pred = np.argmax(previsoes, axis=1)
y_true = dataset_teste.classes
class_names = list(dataset_teste.class_indices.keys())

print(f"Acurácia: {accuracy_score(y_true, y_pred):.2%}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Confusion Matrix")
plt.ylabel("Real")
plt.xlabel("Previsto")
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
print(classification_report(y_true, y_pred, target_names=class_names))